# Análisis — `test_tradetrack_out.parquet`

Exploración de tipos de datos, calidad y perfilado de columnas del export de **TradeTrack**
(trazabilidad de exportación de fruta fresca). Acompaña a `DICCIONARIO_test_tradetrack_out.md`.

Reutilizable: cuando lleguen datos reales, solo cambia `PARQUET_PATH` y re-ejecuta.

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

PARQUET_PATH = 'test_tradetrack_out.parquet'
df = pd.read_parquet(PARQUET_PATH)
print('Shape:', df.shape)
df.head()

## 1. Tipos de datos actuales

In [ ]:
df.dtypes.value_counts()

## 2. Perfilado por columna

No nulos, % nulos, valores únicos y ejemplos.

In [ ]:
def profile(df):
    rows = []
    n = len(df)
    for c in df.columns:
        s = df[c]
        ej = s.dropna().unique()[:3]
        rows.append({
            'columna': c,
            'dtype': str(s.dtype),
            'no_nulos': int(s.notna().sum()),
            'pct_nulos': round(100 * s.isna().mean(), 1),
            'unicos': int(s.nunique(dropna=True)),
            'ejemplos': ', '.join(map(str, ej)),
        })
    return pd.DataFrame(rows)

perfil = profile(df)
perfil

## 3. Columnas numéricas-en-texto y fechas-en-texto

Detección automática de qué columnas `object` parecen numéricas o fechas.

In [ ]:
import re
num_re = re.compile(r'^-?\d+(\.\d+)?$')
date_re = re.compile(r'^\d{4}-\d{2}-\d{2}')

def share(s, rx):
    s = s.dropna().astype(str).str.strip()
    if len(s) == 0:
        return 0.0
    return round(s.map(lambda x: bool(rx.match(x))).mean(), 2)

diag = pd.DataFrame({
    'columna': df.columns,
    'pct_numerico': [share(df[c], num_re) for c in df.columns],
    'pct_fecha_iso': [share(df[c], date_re) for c in df.columns],
})
print('--- Parecen NUMÉRICAS (>=60%) ---')
display(diag[diag.pct_numerico >= 0.6].sort_values('pct_numerico', ascending=False))
print('--- Parecen FECHAS ISO (>=60%) ---')
display(diag[diag.pct_fecha_iso >= 0.6].sort_values('pct_fecha_iso', ascending=False))

## 4. Casteo de tipos sugerido

Esquema explícito basado en el diccionario. Usa `Int64`/`Float64`/`boolean` (nullable) para tolerar nulos.

In [ ]:
cols_int = ['SEMANA_DE_DESPACHO', 'SEMANA_OCULTO', 'CAJASPALLET', 'CANT_CAJAS_TOTAL',
            'CANT_PALLET_TOTAL', 'CALIBRE', 'NUMERO_DE_PALLETS_GR', 'CANTIDADES_DE_CAJAS_PL',
            'CANTIDAD_DE_CAJAS_FACT']
cols_float = ['PESOCAJA', 'KILOS_NETOS_PL', 'KILOS_BRUTOS_PL', 'CANTIDAD_DE_KG_FUNDO_01',
              'CANTIDAD_DE_KG_FUNDO_02', 'CANTIDAD_DE_KG_FUNDO_03', 'TOTAL_FACTURA',
              'KILOS_NETOS_FACTURA', 'Total_de_Liquidacion', 'Liquidacion_x_Unidad',
              'FLETE_AEREO', 'VAT']
cols_fecha_iso = ['ETD', 'ETA', 'FECHA_DE_DESPACHO', 'FECHA_DE_RETIRO', 'FECHA_PROBABLE_DESPACHO',
                  'FECHA_DE_CARGA__DAM_PROV', 'FECHA_DE_CARGA_DAM_REG', 'FECHA_DE_CARGA_DAM_REC',
                  'EMISION_DE_NOTA_CONTABLE']
cols_fecha_dmy = ['FECHA_DE_DESPACHO_OCULTO']
cols_datetime = ['FECHA_Y_HORA']
cols_bool = ['CO_CORRECTO']

out = df.copy()
for c in cols_int:
    if c in out:
        out[c] = pd.to_numeric(out[c], errors='coerce').astype('Int64')
for c in cols_float:
    if c in out:
        out[c] = pd.to_numeric(out[c], errors='coerce').astype('Float64')
for c in cols_fecha_iso + cols_datetime:
    if c in out:
        out[c] = pd.to_datetime(out[c], errors='coerce')
for c in cols_fecha_dmy:
    if c in out:
        out[c] = pd.to_datetime(out[c], errors='coerce', dayfirst=True)
for c in cols_bool:
    if c in out:
        out[c] = out[c].astype(str).str.strip().str.lower().map({'true': True, 'false': False}).astype('boolean')

out.dtypes.value_counts()

In [ ]:
# Estadísticas tras el casteo
out.describe(include='number').T

## 5. Chequeos de calidad

In [ ]:
# Columnas casi vacías
print('Columnas con >80% de nulos:')
display(perfil[perfil.pct_nulos > 80][['columna', 'pct_nulos']])

# Posibles duplicados de contenido
pares = [('MODALIDAD_DE_TRANSPORTE', 'MODALIDAD'),
         ('PUERTO_DESTINO', 'PUERTO_DESTINO_1'),
         ('SEMANA_DE_DESPACHO', 'SEMANA_OCULTO')]
for a, b in pares:
    if a in df and b in df:
        igual = (df[a].fillna('') == df[b].fillna('')).mean()
        print(f'{a} == {b}: {igual:.0%} de filas iguales')

## 6. Vistas de negocio (ejemplos)

In [ ]:
print('Pedidos por producto:')
display(df['DESCRIPCION'].value_counts(dropna=False))
print('\nPedidos por modalidad:')
display(df['MODALIDAD_DE_TRANSPORTE'].value_counts(dropna=False))
print('\nCajas totales por cliente final:')
display(out.groupby('CLIENTE_FINAL')['CANT_CAJAS_TOTAL'].sum().sort_values(ascending=False))